# OpenCV Live-Demo: Müllklassifikation mit 3 Modellen

## Was dieses Notebook macht

Sie können Ihre drei trainierten Modelle live mit der Webcam testen
und mit der Tastatur zwischen ihnen umschalten:

- **1** → MLP
- **2** → Eigenes CNN
- **3** → EfficientNetB0
- **q** → Beenden

---

## Wichtiger Hinweis

Dieses Notebook lädt die Modelle **direkt aus den `.keras`-Dateien**.

Das ist bequem, weil die komplette Architektur und die Gewichte bereits in einer Datei enthalten sind.
Voraussetzung ist aber, dass die `.keras`-Dateien mit einer kompatiblen Keras-/TensorFlow-Version gespeichert wurden.

Falls das direkte Laden einmal nicht funktioniert, wäre die robustere Alternative:
- Architektur im Code definieren
- anschließend nur die Gewichte laden

Für dieses Demo verwenden wir jetzt bewusst den **direkten `.keras`-Import**.

---
## Benötigte Dateien

Dieses Notebook erwartet im selben Ordner:

```text
modell_01_mlp.keras
modell_02_cnn.keras
modell_03_efficientnetb0.keras
model_config.json
```

---
## Schritt 1 – Bibliotheken laden

In [1]:
import cv2
import json
import numpy as np
import tensorflow as tf
from pathlib import Path

print("TensorFlow:", tf.__version__)
print("Keras     :", tf.keras.__version__)
print("OpenCV    :", cv2.__version__)
print("Bibliotheken geladen.")

TensorFlow: 2.21.0
Keras     : 3.15.0
OpenCV    : 5.0.0
Bibliotheken geladen.


---
## Schritt 2 – Dateien prüfen und Konfiguration laden

`model_config.json` enthält:
- Anzahl Klassen
- Bildgrößen
- Klassennamen

In [2]:
MODEL_DIR = Path(".")

MODEL_FILES = {
    1: MODEL_DIR / "modell_01_mlp.keras",
    2: MODEL_DIR / "modell_02_cnn.keras",
    3: MODEL_DIR / "modell_03_efficientnetb0.keras",
}

MODEL_LABELS = {
    1: "MLP",
    2: "Eigenes CNN",
    3: "EfficientNetB0",
}

fehlend = [str(p) for p in MODEL_FILES.values() if not p.exists()]
if not (MODEL_DIR / "model_config.json").exists():
    fehlend.append("model_config.json")

if fehlend:
    raise FileNotFoundError(
        "Fehlende Dateien – bitte zuerst das Trainings-Notebook vollständig ausführen:\n"
        + "\n".join(fehlend)
    )

with open(MODEL_DIR / "model_config.json", "r", encoding="utf-8") as f:
    cfg = json.load(f)

num_classes       = cfg["num_classes"]
IMG_SIZE_SMALL    = tuple(cfg["img_size_small"])
IMG_SIZE_TRANSFER = tuple(cfg["img_size_transfer"])
class_names       = cfg["class_names"]

print("Konfiguration geladen:")
print(f"  Anzahl Klassen : {num_classes}")
print(f"  Klassen        : {class_names}")
print(f"  Größe MLP/CNN  : {IMG_SIZE_SMALL}")
print(f"  Größe Transfer : {IMG_SIZE_TRANSFER}")

Konfiguration geladen:
  Anzahl Klassen : 12
  Klassen        : ['battery', 'biological', 'brown-glass', 'cardboard', 'clothes', 'green-glass', 'metal', 'paper', 'plastic', 'shoes', 'trash', 'white-glass']
  Größe MLP/CNN  : (128, 128)
  Größe Transfer : (224, 224)


---
## Schritt 3 – Modelle direkt aus `.keras` laden

Hier verwenden wir den einfachsten Fall:

```python
tf.keras.models.load_model(...)
```

Die Modelle enthalten bereits:
- Architektur
- Gewichte
- Konfiguration

Deshalb müssen wir sie nicht selbst neu aufbauen.

In [3]:
models = {}
input_sizes = {}

for key in [1, 2, 3]:
    path  = MODEL_FILES[key]
    label = MODEL_LABELS[key]
    print(f"Lade Modell {key}: {label} ...")

    model = tf.keras.models.load_model(path)
    models[key] = model

    _, h, w, _ = model.input_shape
    input_sizes[key] = (h, w)

    print(f"  -> Eingabegröße: {w}x{h}")

print("\nAlle Modelle erfolgreich geladen.")

Lade Modell 1: MLP ...
  -> Eingabegröße: 128x128
Lade Modell 2: Eigenes CNN ...
  -> Eingabegröße: 128x128
Lade Modell 3: EfficientNetB0 ...
  -> Eingabegröße: 224x224

Alle Modelle erfolgreich geladen.


---
## Schritt 4 – Hilfsfunktionen

### `prepare_frame(...)`
Bereitet ein Kamerabild für das Modell vor:
- OpenCV liefert Bilder in **BGR**
- Modelle erwarten hier **RGB**
- das Bild wird auf die richtige Größe gebracht
- daraus wird ein Tensor der Form `(1, H, W, 3)`

### `predict_topk(...)`
Das Modell gibt für jede Klasse eine Wahrscheinlichkeit aus.
Die höchste Wahrscheinlichkeit ist die eigentliche Vorhersage.
Die Top-3 helfen uns zu sehen, welche Alternativen das Modell außerdem noch plausibel fand.

In [4]:
def prepare_frame(frame_bgr, target_h, target_w):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    resized   = cv2.resize(frame_rgb, (target_w, target_h), interpolation=cv2.INTER_AREA)
    x = resized.astype(np.float32)
    x = np.expand_dims(x, axis=0)
    return x


def predict_topk(model, x, class_names, k=3):
    probs      = model.predict(x, verbose=0)[0]
    pred_idx   = int(np.argmax(probs))
    pred_class = class_names[pred_idx]
    pred_conf  = float(probs[pred_idx]) * 100.0
    topk_idx   = np.argsort(probs)[-k:][::-1]
    topk       = [(class_names[i], float(probs[i]) * 100.0) for i in topk_idx]
    return pred_class, pred_conf, topk


print("Hilfsfunktionen bereit.")

Hilfsfunktionen bereit.


---
## Schritt 5 – Live-Demo starten

### Tastensteuerung
- **1** = MLP
- **2** = CNN
- **3** = EfficientNetB0
- **q** = Beenden

### Was im Fenster angezeigt wird
- aktives Modell
- Eingabegröße
- beste Vorhersage mit Konfidenz
- Top-3-Klassen

> **Hinweis:** Falls die Webcam nicht öffnet,
> ändern Sie `CAMERA_INDEX` auf `1` oder `2`.

In [5]:
active_key   = 3
CAMERA_INDEX = 0

cap = cv2.VideoCapture(CAMERA_INDEX)
if not cap.isOpened():
    raise RuntimeError(
        "Webcam konnte nicht geöffnet werden.\n"
        "Tipp: CAMERA_INDEX auf 1 oder 2 ändern."
    )

print("Live-Demo gestartet. Tasten: 1/2/3, q zum Beenden")

while True:
    ok, frame = cap.read()
    if not ok:
        print("Kamerabild konnte nicht gelesen werden.")
        break

    model      = models[active_key]
    model_name = MODEL_LABELS[active_key]
    target_h, target_w = input_sizes[active_key]

    x = prepare_frame(frame, target_h, target_w)
    pred_class, pred_conf, top3 = predict_topk(model, x, class_names, k=3)

    display = frame.copy()
    cv2.rectangle(display, (0, 0), (display.shape[1], 190), (20, 20, 20), -1)

    cv2.putText(display, f"Modell: {model_name} ({target_w}x{target_h})",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 0), 2, cv2.LINE_AA)
    cv2.putText(display, f"Vorhersage: {pred_class} ({pred_conf:.1f}%)",
                (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.80, (0, 255, 0), 2, cv2.LINE_AA)

    y0 = 100
    for i, (name, conf) in enumerate(top3, start=1):
        cv2.putText(display, f"Top {i}: {name} ({conf:.1f}%)",
                    (10, y0 + (i - 1) * 28), cv2.FONT_HERSHEY_SIMPLEX, 0.65,
                    (255, 255, 255), 2, cv2.LINE_AA)

    cv2.putText(display, "Tasten: 1=MLP  2=CNN  3=EfficientNetB0  q=Beenden",
                (10, display.shape[0] - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.58,
                (220, 220, 220), 1, cv2.LINE_AA)

    cv2.imshow("Live-Muellklassifikation", display)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('1'):
        active_key = 1
    elif key == ord('2'):
        active_key = 2
    elif key == ord('3'):
        active_key = 3

cap.release()
cv2.destroyAllWindows()
print("Demo beendet.")

Live-Demo gestartet. Tasten: 1/2/3, q zum Beenden
Demo beendet.


---
## Fazit

Mit diesem Demo-Notebook haben Sie jetzt einen kompletten Live-Workflow mit direktem `.keras`-Laden:

- Modelle direkt aus `.keras` laden
- Live-Vorhersage mit OpenCV
- Top-3-Ausgabe
- Modellwechsel per Tastatur

Wenn alle drei `.keras`-Dateien erfolgreich geladen werden, ist dies die bequemste Variante.